In [1]:

import logging
from pathlib import Path

from pathling import PathlingContext
from pathling._version import __delta_version__, __java_version__, __scala_version__
from pyspark.sql import DataFrame, SparkSession
import pyspark.sql.functions as sf

In [20]:
! pip freeze | grep pathling

IOStream.flush timed out
pathling==8.1.0


In [2]:
logging.basicConfig(level=logging.INFO, format="%(message)s")
_LOGGER = logging.getLogger(__name__)


def _spark_session(ivy_dir: Path) -> SparkSession:
    packages = (
        f"au.csiro.pathling:library-runtime:{__java_version__},"
        f"io.delta:delta-spark_{__scala_version__}:{__delta_version__},"
    )
    return (
        SparkSession.builder
        .appName("pathling-fhir-source-loader")
        .master("local[*]")
        .config("spark.jars.ivy", str(ivy_dir))
        .config("spark.jars.packages", packages)
        .config("spark.sql.shuffle.partitions", "4")
        .getOrCreate()
    )


def _read_resource_lines(spark: SparkSession, path: Path) -> DataFrame:
    return (
        spark.read.option("header", "true")
        .option("multiLine", "false")
        .option("quote", '"')
        .option("escape", '"')
        .csv(str(path))
        .select("line")
        .filter("line IS NOT NULL")
    )


def _load_fhir_resource(pc: PathlingContext, path: Path) -> int:
    base = path.name[: -len("_fhir_raw.csv")]  # type: ignore[assignment]
    resource_name = base if base[0].isupper() else base.capitalize()
    df = _read_resource_lines(pc.spark, path)
    encoded = pc.encode(df, resource_name, column="line")
    encoded.createOrReplaceTempView(resource_name.lower())
    count = encoded.count()
    _LOGGER.info("encoded %-25s rows=%d", resource_name, count)
    return resource_name, encoded





In [3]:
__file__ = "/Users/li-hengfu/Documents/GitHub/demo-prior-auth/"
base_dir = Path(__file__).resolve()
data_dir = base_dir / "fhir_source"
ivy_dir = base_dir / ".ivy2"
ivy_dir.mkdir(exist_ok=True)
spark = _spark_session(ivy_dir)
pc = PathlingContext.create(spark)

fhir_raw_files = sorted(data_dir.glob("*_fhir_raw.csv"))

:: loading settings :: url = jar:file:/Users/li-hengfu/Documents/env/fhir/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/li-hengfu/Documents/GitHub/demo-prior-auth/.ivy2/cache
The jars for the packages stored in: /Users/li-hengfu/Documents/GitHub/demo-prior-auth/.ivy2/jars
au.csiro.pathling#library-runtime added as a dependency
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-df92cb49-7ca6-4ab0-a3d8-387ce4311f69;1.0
	confs: [default]
	found au.csiro.pathling#library-runtime;8.1.0 in central
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 2854ms :: artifacts dl 39ms
	:: modules in use:
	au.csiro.pathling#library-runtime;8.1.0 from central in [default]
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	------------------------------------------------------------

In [17]:
fhir_raw_files

[PosixPath('/Users/li-hengfu/Documents/GitHub/demo-prior-auth/fhir_source/MedicationAdministration_fhir_raw.csv'),
 PosixPath('/Users/li-hengfu/Documents/GitHub/demo-prior-auth/fhir_source/MedicationDispense_fhir_raw.csv'),
 PosixPath('/Users/li-hengfu/Documents/GitHub/demo-prior-auth/fhir_source/MedicationRequest_fhir_raw.csv'),
 PosixPath('/Users/li-hengfu/Documents/GitHub/demo-prior-auth/fhir_source/MedicationStatement_fhir_raw.csv'),
 PosixPath('/Users/li-hengfu/Documents/GitHub/demo-prior-auth/fhir_source/condition_fhir_raw.csv'),
 PosixPath('/Users/li-hengfu/Documents/GitHub/demo-prior-auth/fhir_source/encounter_fhir_raw.csv'),
 PosixPath('/Users/li-hengfu/Documents/GitHub/demo-prior-auth/fhir_source/location_fhir_raw.csv'),
 PosixPath('/Users/li-hengfu/Documents/GitHub/demo-prior-auth/fhir_source/medication_fhir_raw.csv'),
 PosixPath('/Users/li-hengfu/Documents/GitHub/demo-prior-auth/fhir_source/observation_fhir_raw.csv'),
 PosixPath('/Users/li-hengfu/Documents/GitHub/demo-prior

In [4]:
fhir_sources = {}

_LOGGER.info("loading %d FHIR resource files", len(fhir_raw_files))

for resource_path in fhir_raw_files:
    resource_name, encoded= _load_fhir_resource(pc, resource_path)
    fhir_sources[resource_name] = encoded
table_names = [table.name for table in spark.catalog.listTables()]
_LOGGER.info("Pathling views: %s", ", ".join(sorted(table_names)))

data = pc.read.datasets(fhir_sources)


loading 13 FHIR resource files
26/04/16 12:22:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
encoded MedicationAdministration  rows=56535                                    
encoded MedicationDispense        rows=15375                                    
encoded MedicationRequest         rows=17552                                    
encoded MedicationStatement       rows=2411
encoded Condition                 rows=5051
encoded Encounter                 rows=637
encoded Location                  rows=31
encoded Medication                rows=1794
encoded Observation               rows=813540                                   
encoded Organization              rows=1
encoded Patient                   rows=100
encoded Procedure                 rows=3450
encoded Specimen                  rows=12458
Pathling views: condition, encounter, location, medication, medica

In [26]:
help(data)

Help on DataSource in module pathling.datasource object:

class DataSource(pathling.core.SparkConversionsMixin)
 |  DataSource(jds: py4j.java_gateway.JavaObject, pc: pathling.context.PathlingContext)
 |  
 |  A data source that can be used to run queries against FHIR data.
 |  
 |  Method resolution order:
 |      DataSource
 |      pathling.core.SparkConversionsMixin
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(self, jds: py4j.java_gateway.JavaObject, pc: pathling.context.PathlingContext)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  read(self, resource_code: str) -> pyspark.sql.dataframe.DataFrame
 |      Reads the data for the given resource type from the data source.
 |      
 |      :param resource_code: A string representing the type of FHIR resource to read data from.
 |      
 |      :return: A Spark DataFrame containing the data for the given resource type.
 |  
 |  resource_types(self)
 |      Returns a list of the

In [5]:
patient_table = data.view(
    resource="Patient",
    select=[
        {
            "column": [
                {"path": "id", "name": "subject_id"},
                {"path": "identifier.value", "name": "subject_source_id"},
                {"path": "gender", "name": "gender"},
                {"path": "birthDate", "name": "birth_date"},
            ]
        }
    ],
)

patient_table.show(5, False)

+------------------------------------+-----------------+------+----------+
|subject_id                          |subject_source_id|gender|birth_date|
+------------------------------------+-----------------+------+----------+
|28dcf33b-0c52-587f-83ad-2a3270976719|10007795         |female|2083-04-10|
|74a2fd87-885b-5eca-9f8b-9141915dba51|10007928         |female|2070-04-05|
|51d2190c-cc46-56c5-b2ea-363895cbea75|10009628         |male  |2095-09-17|
|c4c29979-f2f5-56db-af5b-1715887727b8|10011398         |male  |2079-12-15|
|b410dd44-7d65-56f9-974f-2751e8aa80e2|10004457         |male  |2075-09-17|
+------------------------------------+-----------------+------+----------+
only showing top 5 rows



In [6]:
conditions_table = data.view(
    resource="Condition",
    select=[
        {
            "column": [
                {"path": "id", "name": "condition_id"},
                {"path": "subject.reference", "name": "raw_patient_reference"},
                {"path": "encounter.reference", "name": "raw_encounter_reference"},
                {"path": "code.coding.code", "name": "raw_condition_code"},
                {"path": "code.coding.system", "name": "raw_code_class"},
                {"path": "code.coding.display", "name": "condition_name"},
            ]
        }
    ],
)

conditions_table.show(5, False)

+------------------------------------+--------------------------------------------+----------------------------------------------+------------------+---------------------------------------------------------------+-------------------------------------------------+
|condition_id                        |raw_patient_reference                       |raw_encounter_reference                       |raw_condition_code|raw_code_class                                                 |condition_name                                   |
+------------------------------------+--------------------------------------------+----------------------------------------------+------------------+---------------------------------------------------------------+-------------------------------------------------+
|00934dbb-5139-57fe-aca5-e26de86605c4|Patient/0a8eebfd-a352-522e-89f0-1d4a13abdebc|Encounter/8a5be724-d9d4-5a47-8a39-4a274662f766|V462              |http://mimic.mit.edu/fhir/mimic/CodeSystem/mimic-diagnosis-

In [ ]:
encounter_view = data.view(
    resource="Encounter",
    select=[
        {
            "column": [
                {"path": "id", "name": "encounter_id"},
                {
                    "path": "type.coding.display.first()",
                    "name": "encounter_type",
                },
                {
                    "path": "class.display.first()",
                    "name": "encounter_class",
                },
                {"path": "period.start", "name": "raw_start_datetime"},
                {"path": "period.end", "name": "raw_end_datetime"},
                {"path": "subject.reference", "name": "raw_patient_reference"},
                {
                    "path": "serviceType.coding.code.first()",
                    "name": "service_type",
                },
                {
                    "path": "hospitalization.admitSource.coding.code.first()",
                    "name": "admission_source",
                },
                {
                    "path": "hospitalization.dischargeDisposition.coding.code.first()",
                    "name": "discharge_disposition",
                },
                {"path": "serviceProvider.reference", "name": "raw_service_provider"},
                {"path": "partOf.reference", "name": "associated_parent_encounter_id"},
                {"path": "priority.coding.code.first()", "name": "priority_code"},
                {"path": "identifier.system.first()", "name": "raw_encounter_identifier"}
            ]
        }
    ],
)



encounter_view = (
    encounter_view
    .withColumn(
        "encounter_start_datetime",
        sf.to_timestamp(sf.col("raw_start_datetime"))
    )
    .withColumn(
        "encounter_end_datetime",
        sf.to_timestamp(sf.col("raw_end_datetime"))
    )
    .withColumn(
        "encounter_identifier", 
        sf.split(sf.col("raw_encounter_identifier"), "-")
        .getItem(1)
    )
    .withColumn(
        "subject_id", 
        sf.split(sf.col("raw_patient_reference"), "/")
        .getItem(1)
    )
    .withColumn(
        "service_provider", 
        sf.split(sf.col("raw_service_provider"), "/")
        .getItem(1)
    )
    .drop("raw_patient_reference")
    .drop("raw_service_provider")
    .drop("raw_encounter_identifier")
)

encounter_view.show()

+--------------------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+---------------------+--------------------+-------------+------------------------+----------------------+--------------------+--------------------+--------------------+
|        encounter_id|      encounter_type|     encounter_class|  raw_start_datetime|    raw_end_datetime|service_type|    admission_source|discharge_disposition|raw_parent_encounter|priority_code|encounter_start_datetime|encounter_end_datetime|encounter_identifier|          subject_id|    service_provider|
+--------------------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+---------------------+--------------------+-------------+------------------------+----------------------+--------------------+--------------------+--------------------+
|9c4ef2ae-dd61-5ef...|Patient encounter...|          ambulatory|2180-05-0

In [5]:
medication_request_view = data.view(
    resource="MedicationRequest",
    select=[
        {
            "column": [
                {"path": "id", "name": "med_request_id"},
                {"path": "identifier.first().value", "name": "phid"},
                {"path": "status", "name": "status"},
                {"path": "intent", "name": "intent"},
                {"path": "subject.reference", "name": "patient_reference"},
                {"path": "encounter.reference", "name": "encounter_reference"},
                {"path": "authoredOn", "name": "authored_on"},
                {"path": "dispenseRequest.validityPeriod.start", "name": "dispense_start"},
                {"path": "dispenseRequest.validityPeriod.end", "name": "dispense_end"},
                {"path": "medicationReference.reference", "name": "medication_reference"},
            ]
        },
        {
            "forEach": "dosageInstruction",
            "column": [
                {"path": "text", "name": "dosage_text"},
                {
                    "path": "route.coding.first().code",
                    "name": "route_code",
                },
                {
                    "path": "route.coding.first().system",
                    "name": "route_system",
                },
                {
                    "path": "timing.code.coding.first().code",
                    "name": "frequency_code",
                },
                {
                    "path": "doseAndRate.first().doseQuantity.value",
                    "name": "dose_value",
                },
                {
                    "path": "doseAndRate.first().doseQuantity.unit",
                    "name": "dose_unit",
                },
            ],
        },
    ],
)


medication_request_view.show()

26/04/07 15:51:49 WARN Collection: Traversing a choice element `medicationReference` without using ofType() is not portable and may not work in some FHIRPath implementations. Consider using ofType() to specify the type of element you want to traverse.
26/04/07 15:51:49 WARN Collection: Traversing a choice element `doseQuantity` without using ofType() is not portable and may not work in some FHIRPath implementations. Consider using ofType() to specify the type of element you want to traverse.
26/04/07 15:51:49 WARN Collection: Traversing a choice element `doseQuantity` without using ofType() is not portable and may not work in some FHIRPath implementations. Consider using ofType() to specify the type of element you want to traverse.
26/04/07 15:51:49 WARN Collection: Traversing a choice element `doseQuantity` without using ofType() is not portable and may not work in some FHIRPath implementations. Consider using ofType() to specify the type of element you want to traverse.
26/04/07 15:5

+--------------------+--------+---------+------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------------+----------+---------+
|      med_request_id|    phid|   status|intent|   patient_reference| encounter_reference|         authored_on|      dispense_start|        dispense_end|medication_reference|         dosage_text|route_code|        route_system|frequency_code|dose_value|dose_unit|
+--------------------+--------+---------+------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------------+----------+---------+
|1265e24c-e1f0-5b1...|53072793|completed| order|Patient/0a8eebfd-...|Encounter/8f0d275...|2180-08-06T08:06:...|2180-08-06T09:00:...|2180-08-06T11:00:...|Medication/83d477...|     15g/60mL Bottle|     PO/NG|ht

In [ ]:
medication_df = data.view(
    resource="Medication",
    select=[
        {
            "column": [
                {"path": "id", "name": "medication_id"},
                {"path": "status", "name": "status"},

                {
                    "path": "code.coding.where(system = 'http://mimic.mit.edu/fhir/mimic/CodeSystem/mimic-medication-ndc').code.first()",
                    "name": "ndc_code",
                },
                {
                    "path": "code.coding.where(system = 'http://mimic.mit.edu/fhir/mimic/CodeSystem/mimic-medication-formulary-drug-cd').code.first()",
                    "name": "formulary_drug_cd",
                },

                {
                    "path": "identifier.where(system = 'http://mimic.mit.edu/fhir/mimic/CodeSystem/mimic-medication-ndc').value.first()",
                    "name": "ndc_identifier",
                },
                {
                    "path": "identifier.where(system = 'http://mimic.mit.edu/fhir/mimic/CodeSystem/mimic-medication-formulary-drug-cd').value.first()",
                    "name": "formulary_identifier",
                },
                {
                    "path": "identifier.where(system = 'http://mimic.mit.edu/fhir/mimic/CodeSystem/mimic-medication-name').value.first()",
                    "name": "medication_name",
                },
                {
                    "path": "identifier.where(system = 'http://mimic.mit.edu/fhir/mimic/identifier/medication-mix').value.first()",
                    "name": "medication_mix_identifier",
                }
                # {
                #     "path": "ingredient.itemReference.reference.first()",
                #     "name": "ingredient_1_reference",
                # },
                # {
                #     "path": "ingredient.itemReference.reference.skip(1).first()",
                #     "name": "ingredient_2_reference",
                # }
            ]
        }
    ],
)


medication_df.show()

Py4JJavaError: An error occurred while calling o576.execute.
: au.csiro.pathling.errors.UnsupportedFhirPathFeatureError: Unsupported function: coalesce
	at au.csiro.pathling.fhirpath.path.Paths$EvalFunction.apply(Paths.java:196)
	at au.csiro.pathling.fhirpath.execution.FhirpathEvaluator.evaluate(FhirpathEvaluator.java:127)
	at au.csiro.pathling.projection.ProjectionContext.evalExpression(ProjectionContext.java:75)
	at au.csiro.pathling.projection.ColumnSelection.lambda$getCollectionIterator$1(ColumnSelection.java:77)
	at java.base/java.util.stream.ReferencePipeline$3$1.accept(ReferencePipeline.java:197)
	at java.base/java.util.AbstractList$RandomAccessSpliterator.tryAdvance(AbstractList.java:708)
	at java.base/java.util.stream.StreamSpliterators$WrappingSpliterator.lambda$initPartialTraversalState$0(StreamSpliterators.java:292)
	at java.base/java.util.stream.StreamSpliterators$AbstractWrappingSpliterator.fillBuffer(StreamSpliterators.java:206)
	at java.base/java.util.stream.StreamSplit

In [26]:
medication_base = data.view(
    resource="Medication",
    select=[
        {
            "column": [
                {"path": "id", "name": "medication_id"},
                {"path": "status", "name": "status"},

                # Coding-based fields
                {
                    "path": "code.coding.where(system = 'http://mimic.mit.edu/fhir/mimic/CodeSystem/mimic-medication-ndc').code.first()",
                    "name": "ndc_code",
                },
                {
                    "path": "code.coding.where(system = 'http://mimic.mit.edu/fhir/mimic/CodeSystem/mimic-medication-formulary-drug-cd').code.first()",
                    "name": "formulary_drug_cd",
                },

                # Identifier-based fields
                {
                    "path": "identifier.where(system = 'http://mimic.mit.edu/fhir/mimic/CodeSystem/mimic-medication-ndc').value.first()",
                    "name": "ndc_identifier",
                },
                {
                    "path": "identifier.where(system = 'http://mimic.mit.edu/fhir/mimic/CodeSystem/mimic-medication-formulary-drug-cd').value.first()",
                    "name": "formulary_identifier",
                },
                {
                    "path": "identifier.where(system = 'http://mimic.mit.edu/fhir/mimic/CodeSystem/mimic-medication-name').value.first()",
                    "name": "medication_name",
                },
                {
                    "path": "identifier.where(system = 'http://mimic.mit.edu/fhir/mimic/identifier/medication-mix').value.first()",
                    "name": "medication_mix_identifier",
                },
            ]
        }
    ],
)

medication_base = (
    medication_base
    .withColumn(
        "is_compounded",
        sf.when(
            sf.col("medication_mix_identifier").isNotNull(),
            sf.lit(True)
        ).otherwise(sf.lit(False))
    )
    .withColumn(
        "preferred_medication_label",
        sf.coalesce(
            sf.col("medication_name"),
            sf.col("medication_mix_identifier")
        )
    )
)


medication_base.show()

+--------------------+------+-----------+-----------------+--------------+--------------------+--------------------+-------------------------+-------------+--------------------------+
|       medication_id|status|   ndc_code|formulary_drug_cd|ndc_identifier|formulary_identifier|     medication_name|medication_mix_identifier|is_compounded|preferred_medication_label|
+--------------------+------+-----------+-----------------+--------------+--------------------+--------------------+-------------------------+-------------+--------------------------+
|ed5a988e-5fb0-5e1...|active|51079030020|             NULL|   51079030020|               CATA2|           CloniDINE|                     NULL|        false|                 CloniDINE|
|dfa3853a-044c-5d9...|active|63323037601|             NULL|   63323037601|            OCTR0.1I|  Octreotide Acetate|                     NULL|        false|        Octreotide Acetate|
|45759ff7-6e2d-54a...|active|23155029442|             NULL|   23155029442|      

In [20]:
medication_ingredient_rows = data.view(
    resource="Medication",
    select=[
        {
            "column": [
                {"path": "id", "name": "medication_id"},
            ]
        },
        {
            "forEach": "ingredient",
            "column": [
                {
                    "path": "item.ofType(Reference).reference",
                    "name": "ingredient_reference",
                }
            ],
        },
    ],
)

medication_ingredient_rows = (
    medication_ingredient_rows
    .withColumn(
        "ingredient_medication_id",
        sf.regexp_extract(sf.col("ingredient_reference"), r"^Medication/(.+)$", 1)
    )
)

medication_ingredient_rows.show()

+--------------------+--------------------+------------------------+
|       medication_id|ingredient_reference|ingredient_medication_id|
+--------------------+--------------------+------------------------+
|2eda3bf3-1e8f-517...|Medication/f84ef2...|    f84ef276-2687-5ea...|
|2eda3bf3-1e8f-517...|Medication/7c46a1...|    7c46a14c-af29-58d...|
|1e978837-9bed-518...|Medication/f25cf5...|    f25cf585-8c8a-5f1...|
|1e978837-9bed-518...|Medication/6378e7...|    6378e7ba-a958-5a0...|
|effa458d-ac56-510...|Medication/1750aa...|    1750aac2-1c79-549...|
|effa458d-ac56-510...|Medication/093303...|    09330373-5f68-58c...|
|de2acfa2-af07-576...|Medication/039f77...|    039f7713-5399-561...|
|de2acfa2-af07-576...|Medication/2b2d3a...|    2b2d3a39-ca72-5df...|
|959d5cbd-92fa-5eb...|Medication/e87e1d...|    e87e1d0c-91a1-563...|
|959d5cbd-92fa-5eb...|Medication/e5bc66...|    e5bc6623-65b7-518...|
|ccde1572-f635-557...|Medication/148ca7...|    148ca77a-cd68-513...|
|ccde1572-f635-557...|Medication/e

In [32]:
from pyspark.sql import functions as sf

from pyspark.sql import functions as sf

procedure_view = data.view(
    resource="Procedure",
    select=[
        {
            "column": [
                {"path": "id", "name": "procedure_id"},
                {"path": "status", "name": "status"},
                {"path": "subject.reference", "name": "raw_patient_reference"},
                {"path": "encounter.reference", "name": "raw_encounter_reference"},

                {"path": "code.coding.code.first()", "name": "procedure_code"},
                {"path": "code.coding.system.first()", "name": "procedure_code_system"},
                {"path": "code.coding.display.first()", "name": "procedure_display"},

                {
                    "path": "performed.ofType(dateTime)",
                    "name": "performed_datetime",
                },
                {
                    "path": "performed.ofType(Period).start",
                    "name": "performed_period_start",
                },
                {
                    "path": "performed.ofType(Period).end",
                    "name": "performed_period_end",
                },

#                 {
#                     "path": "category.coding.code.first()",
#                     "name": "category_code",
#                 },
#                 {
#                     "path": "category.coding.display.first()",
#                     "name": "category_display_raw",
#                 },
            ]
        }
    ],
)

procedure_view = (
    procedure_view
    .withColumn(
        "subject_id", 
        sf.split(sf.col("raw_patient_reference"), "/")
        .getItem(1)
    )
    .withColumn(
        "encounter_id", 
        sf.split(sf.col("raw_encounter_reference"), "/")
        .getItem(1)
    )
    .withColumn(
        "procedure_datestart",
        sf.to_timestamp(
            sf.coalesce(
                sf.col("performed_datetime"),
                sf.col("performed_period_start")
            )
        )
    )
    .withColumn(
        "procedure_dateend",
        sf.to_timestamp(
            sf.col("performed_period_end"))
    )
    .withColumn(
        "code_class", 
        sf.split(sf.col("procedure_code_system"), "-")
        .getItem(2)
    )
    .drop("raw_patient_reference")
    .drop("raw_encounter_reference")
    .drop("performed_datetime")
    .drop("performed_period_start")
    .drop("performed_period_end")
    .drop("raw_code_class")
)


procedure_view.show()

+--------------------+---------+--------------+---------------------+--------------------+--------------------+--------------------+-------------------+-----------------+----------+
|        procedure_id|   status|procedure_code|procedure_code_system|   procedure_display|          subject_id|        encounter_id|procedure_datestart|procedure_dateend|code_class|
+--------------------+---------+--------------+---------------------+--------------------+--------------------+--------------------+-------------------+-----------------+----------+
|3f3c722c-9839-52f...|completed|          5491| http://mimic.mit....|Percutaneous abdo...|0a8eebfd-a352-522...|89ec6d79-e48d-569...|2180-06-27 06:00:00|             NULL|      icd9|
|0e121eb2-e8e9-541...|completed|          5491| http://mimic.mit....|Percutaneous abdo...|0a8eebfd-a352-522...|9c4ef2ae-dd61-5ef...|2180-05-07 06:00:00|             NULL|      icd9|
|08ea8150-0f9e-5b6...|completed|          5491| http://mimic.mit....|Percutaneous abdo...|

26/04/18 00:25:28 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 3870889 ms exceeds timeout 120000 ms
26/04/18 00:25:29 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/18 00:25:33 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1240)
	at 

In [10]:
specimen_df = data.view(
    resource="Specimen",
    select=[
        {
            "column": [
                {"path": "id", "name": "specimen_id"},
                {"path": "subject.reference", "name": "raw_patient_reference"},
                {"path": "collection.collectedDateTime", "name": "collected_datetime"},
                {"path": "type.coding.code.first()", "name": "specimen_type_code"},
                {"path": "type.coding.display.first()", "name": "specimen_type_display"},
                {"path": "identifier.value.first()", "name": "specimen_identifier"},
                {"path": "identifier.system.first()", "name": "specimen_identifier_system"},
            ]
        }
    ],
)

specimen_df.show()

26/04/07 17:25:28 WARN Collection: Traversing a choice element `collectedDateTime` without using ofType() is not portable and may not work in some FHIRPath implementations. Consider using ofType() to specify the type of element you want to traverse.


+--------------------+---------------------+--------------------+------------------+---------------------+-------------------+--------------------------+
|         specimen_id|raw_patient_reference|  collected_datetime|specimen_type_code|specimen_type_display|specimen_identifier|specimen_identifier_system|
+--------------------+---------------------+--------------------+------------------+---------------------+-------------------+--------------------------+
|3a4db913-6499-57c...| Patient/0a8eebfd-...|2180-05-06T22:25:...|             70012|        BLOOD CULTURE|            8536332|      http://mimic.mit....|
|3aa75117-9552-5b6...| Patient/0a8eebfd-...|2180-03-23T11:51:...|             70017|       SEROLOGY/BLOOD|            1836584|      http://mimic.mit....|
|1e837e5a-cf20-5eb...| Patient/0a8eebfd-...|2180-05-07T00:19:...|             70070|                 SWAB|            5717063|      http://mimic.mit....|
|d5e8605a-d21e-59e...| Patient/0a8eebfd-...|2180-07-23T06:39:...|           

In [11]:
observation_view = data.view(
    resource="Observation",
    select=[
        {
            "column": [
                {"path": "id", "name": "observation_id"},
#                 {"path": "status", "name": "status"},
                {"path": "category.coding.code.first()", "name": "category_code"},
                {"path": "category.coding.display.first()", "name": "category_display"},
                {"path": "code.coding.code.first()", "name": "observation_code"},
                {"path": "code.coding.system.first()", "name": "observation_system"},
                {"path": "code.coding.display.first()", "name": "observation_display"},
                {"path": "subject.reference", "name": "raw_patient_reference"},
                {"path": "encounter.reference", "name": "raw_encounter_reference"},
                {"path": "effectiveDateTime", "name": "effective_datetime"},
                {"path": "issued", "name": "issued_datetime"},
                {"path": "valueQuantity.value", "name": "value_numeric"},
                {"path": "valueQuantity.unit", "name": "value_unit"},
                {"path": "valueQuantity.system", "name": "value_system"},
                {"path": "valueString", "name": "value_text"}
            ]
        }
    ],
)





observation_view = (
    observation_view
    .withColumn(
        "effective_datetime",
        sf.to_timestamp(sf.col("effective_datetime"))
    )
    .withColumn(
        "issued_datetime",
        sf.to_timestamp(sf.col("issued_datetime"))
    )
    .withColumn(
        "subject_id", 
        sf.split(sf.col("raw_patient_reference"), "/")
        .getItem(1)
    )
    .withColumn(
        "encounter_id", 
        sf.split(sf.col("raw_encounter_reference"), "/")
        .getItem(1)
    )
    .drop("authored_on")
    .drop("dispense_end")
    .drop("dispense_start")
    .drop("raw_patient_reference")
    .drop("raw_encounter_reference")
)



observation_df = (
    observation_view
    # .join(anchor_year_df, 'subject_id')
    # .withColumn(
    #     "effective_datetime", 
    #     sf.expr("effective_datetime - make_interval(anchor_year_gap, 0, 0, 0, 0, 0, 0)")
    # )
    # .withColumn(
    #     "issued_datetime", 
    #     sf.expr("issued_datetime - make_interval(anchor_year_gap, 0, 0, 0, 0, 0, 0)")
    # )
    .withColumn(
        "value_unit",
        sf.when(
            (sf.col("observation_display") == "Admission Weight (lbs.)") &
            (sf.col("value_unit").isNull() | (sf.trim(sf.col("value_unit")) == "")),
            sf.lit("lbs")
        ).otherwise(sf.col("value_unit"))
    )
    .withColumn(
        "category_code",
        sf.when(
            (sf.col("category_code") == "Routine Vital Signs") |
            (sf.col("category_code") == "vital-signs"),
            sf.lit("Vital Signs")
        ).otherwise(sf.col("category_code"))
    )
    .withColumn(
        "category_code",
        sf.when(
            (sf.col("category_code") == "Labs"),
            sf.lit("laboratory")
        ).otherwise(sf.col("category_code"))
    )
    .drop("anchor_year_gap")
)

observation_df.show()

26/04/17 14:37:59 WARN Collection: Traversing a choice element `effectiveDateTime` without using ofType() is not portable and may not work in some FHIRPath implementations. Consider using ofType() to specify the type of element you want to traverse.
26/04/17 14:37:59 WARN Collection: Traversing a choice element `valueQuantity` without using ofType() is not portable and may not work in some FHIRPath implementations. Consider using ofType() to specify the type of element you want to traverse.
26/04/17 14:37:59 WARN Collection: Traversing a choice element `valueQuantity` without using ofType() is not portable and may not work in some FHIRPath implementations. Consider using ofType() to specify the type of element you want to traverse.
26/04/17 14:37:59 WARN Collection: Traversing a choice element `valueQuantity` without using ofType() is not portable and may not work in some FHIRPath implementations. Consider using ofType() to specify the type of element you want to traverse.
26/04/17 14:

+--------------------+-------------+----------------+----------------+--------------------+--------------------+-------------------+-------------------+-------------+----------+--------------------+------------------+--------------------+--------------------+
|      observation_id|category_code|category_display|observation_code|  observation_system| observation_display| effective_datetime|    issued_datetime|value_numeric|value_unit|        value_system|        value_text|          subject_id|        encounter_id|
+--------------------+-------------+----------------+----------------+--------------------+--------------------+-------------------+-------------------+-------------+----------+--------------------+------------------+--------------------+--------------------+
|af68c869-661b-516...|  Vital Signs|            NULL|          223761|http://mimic.mit....|Temperature Fahre...|2180-07-23 20:00:00|2180-07-23 20:20:00|         98.7|        °F|http://mimic.mit....|              NULL|0a8